# Model evaluation — which model should describe the corpus?

The pipeline generates metadata with a local model. Nobody has measured whether
that metadata is good, so every claim about the system currently rests on an
assumption. This notebook removes the assumption.

**The plan.** Four things are measurable without any labelled data, and they
narrow eight candidate models to two or three. Only then is human grading time
spent, on the rubric in `docs/description-grading-rubric.md`.

| Measure | Needs labels? | Why it matters |
|---|---|---|
| Schema adherence | No | A model with unreliable JSON is unusable at volume |
| Abstention | No | Confident wrong output manufactures retention signal |
| Self-consistency | No | A model that contradicts itself cannot be audited |
| Cost | No | Decides what is schedulable |
| Agreement with a person | **Yes** | The real question, and the expensive one |

**Constraint:** everything runs on locally hosted open-weight models. The corpus
contains personal email, unreviewed identifiers and FERPA-covered records, so
hosted APIs are excluded — not as a preference, as the reason the architecture
looks like this.


## 1. Setup

Point `REPO` at your clone. The pipeline modules live in `src/workbench/`.


In [ ]:
import sys, json, time, pathlib
REPO = pathlib.Path.home() / "RIMS-Archival-Project-"
sys.path.insert(0, str(REPO / "src"))

from workbench import evaluate as E
print("harness loaded")
print("required keys:", E.REQUIRED_KEYS)


## 2. Where the inputs come from

Real extracted text from a completed run, not synthetic examples. A model that
handles clean prose and fails on a 1993 mail header with mangled encoding is a
model that fails on this corpus.

If you have no manifest to hand, the fallback cell below builds cases from a
few pasted strings so the harness can be exercised before you have server
access.


In [ ]:
# Option A — from a real run (preferred)
MANIFEST = REPO.parent / "runs/final/manifest.parquet"   # adjust

rows = []
if MANIFEST.exists():
    import pyarrow.parquet as pq
    cols = ["filename","s03_lane","s03_text_len","s03_text_sample","layer"]
    have = [c for c in cols if c in pq.read_schema(MANIFEST).names]
    rows = [r for r in pq.read_table(MANIFEST, columns=have).to_pylist()
            if r.get("layer") != 0]
    print(f"{len(rows)} rows from the manifest")
else:
    print("no manifest here — use Option B")


In [ ]:
# Option B — paste a few real extractions if you have no manifest yet
if not rows:
    rows = [
        {"filename":"referee_report.doc","s03_lane":"document","s03_text_len":800,
         "s03_text_sample":"Referee report on the manuscript submitted to the "
                           "Journal of Fluid Mechanics. The authors present a "
                           "model of annular flow. Section 3 requires revision; "
                           "the closure assumption is not justified."},
        {"filename":"listserv.eml","s03_lane":"email","s03_text_len":600,
         "s03_text_sample":"[gutvol-d] Re: proofreading standards. On Mon, "
                           "4 Oct 2004, someone wrote: I think the current "
                           "guidelines are too strict for older texts."},
    ]
print(f"{len(rows)} rows available")


## 3. Build the test set

`build_cases` takes real files and adds **degraded copies** of them — blanked,
truncated mid-word, reduced to a quoted mail header. Those have a known correct
answer ("I cannot describe this") without anyone labelling anything, which is
what makes abstention measurable today.


In [ ]:
cases = E.build_cases(rows, n_real=20)
real = [c for c in cases if not c.expects_abstention]
hard = [c for c in cases if c.expects_abstention]
print(f"{len(cases)} cases: {len(real)} real, {len(hard)} should be declined")
for c in hard[:3]:
    print(f"  {c.case_id:18} {c.note[:50]}")
    print(f"     -> {c.text[:70]!r}")


## 4. Wire up the models

One function per model, taking a `Case` and returning raw text. The harness
never needs to know which serving stack is behind it, so vLLM, Ollama and a
local transformers pipeline all plug in the same way.

Start the server first:

```
OLLAMA_HOST=127.0.0.1:11435 OLLAMA_KEEP_ALIVE=4h ollama serve &
ollama pull llama3.1:8b
ollama pull qwen2.5:7b
```


In [ ]:
import requests

OLLAMA = "http://127.0.0.1:11435"

PROMPT = '''You are cataloguing a file from a university archives.

Describe ONLY what the text below contains. If the text is too short, garbled,
or has no discernible subject, say so — that is a correct and useful answer,
and a confident summary of unreadable text is worse than none.

Reply with ONE JSON object and nothing else:
{"title": "max 10 words, or null", "description": "2-3 sentences",
 "subjects": ["..."], "genre": "correspondence|report|draft|minutes|form|dataset|publication|notes|list traffic|administrative|unknown",
 "readable": true}

--- TEXT ---
%s
--- END ---'''

def ollama_model(model_name, url=OLLAMA, timeout=180):
    def call(case):
        r = requests.post(f"{url}/api/generate", timeout=timeout, json={
            "model": model_name, "prompt": PROMPT % case.text, "stream": False,
            "options": {"temperature": 0, "num_predict": 400}})
        r.raise_for_status()
        return r.json().get("response","")
    return call

CANDIDATES = {
    "llama3.1:8b": ollama_model("llama3.1:8b"),
    # "qwen2.5:7b": ollama_model("qwen2.5:7b"),
    # "gemma2:9b":  ollama_model("gemma2:9b"),
}
print(f"{len(CANDIDATES)} candidates")


## 5. Run

`repeats=2` at temperature 0 is what makes self-consistency measurable. A model
that returns two different titles for the same input cannot be audited, and no
amount of description quality compensates for that.


In [ ]:
results, attempts_by_model = [], {}

for name, fn in CANDIDATES.items():
    print(f"running {name} over {len(cases)} cases x2 ...")
    t0 = time.perf_counter()
    attempts = E.run_model(name, fn, cases, repeats=2)
    attempts_by_model[name] = attempts
    results.append(E.score(attempts, cases))
    print(f"  done in {time.perf_counter()-t0:.0f}s")

print()
print(E.comparison_table(results))


## 6. Read the table

**JSON% and keys%** — anything below about 95 needs constrained decoding or a
repair pass before it is a candidate. This is usually fixable and is not a
verdict on the model.

**declines% and invents%** — the two that matter. `invents%` is the proportion
of unusable inputs the model described anyway. A model at 0% declines and 100%
invents will silently fill your manifest with confident fiction about blank
pages. Prefer a model that over-declines.

**consist%** — below 100 at temperature 0 is a serious finding. Record it.

**hrs/150k** — the whole personal-papers backlog. Compare against the window
you actually have. Remember the measured 6.22× from moving Ollama to vLLM on
identical hardware: a slow model may be a serving problem, not a model
problem.


In [ ]:
for s in results:
    print(json.dumps(s, indent=2))


## 7. Blind grading sheet

The label-free measures narrow the field. This is where a person decides, using
`docs/description-grading-rubric.md`.

Candidates are tagged, not named. Knowing which model wrote a description
changes how it reads, and the point is to find out which is better.


In [ ]:
sheet = E.grading_sheet(attempts_by_model, cases, limit=15)

# The key is saved separately and NOT shown while grading.
key = {}
for row in sheet:
    key.update(row.pop("_key", {}))

out = REPO.parent / "evaluation"
out.mkdir(exist_ok=True)
(out / "grading-sheet.json").write_text(json.dumps(sheet, indent=1))
(out / "grading-key.json").write_text(json.dumps(key, indent=1))
print(f"wrote {out/'grading-sheet.json'} — {len(sheet)} files to grade")
print("the key is in grading-key.json. Do not open it until the grades are in.")


In [ ]:
# grade one at a time
for row in sheet[:2]:
    print("=" * 72)
    print(row["source"])
    print("INPUT:", row["input_opening"][:240].replace("\n"," "))
    for c in row["candidates"]:
        print(f"\n  [{c['tag']}] {c['title']}")
        print(f"        {c['description']}")


## 8. Record the grades

Four dimensions, 0-3 each, graded separately. Do not average them — a
description can be accurate and useless, and a single number hides that.


In [ ]:
grades = {
    # "case_id": {"tag": {"accuracy":3, "genre":2, "specificity":2, "restraint":3}},
}

def summarise(grades, key):
    from collections import defaultdict
    import statistics as st
    per = defaultdict(lambda: defaultdict(list))
    for case_id, tags in grades.items():
        for tag, dims in tags.items():
            for dim, v in dims.items():
                per[key.get(tag, tag)][dim].append(v)
    for model, dims in per.items():
        print(f"\n{model}")
        for dim, vals in dims.items():
            print(f"  {dim:12} median {st.median(vals)}  range {min(vals)}-{max(vals)}  n={len(vals)}")
        zeros = sum(1 for v in dims.get("accuracy", []) if v == 0)
        if zeros:
            print(f"  ** {zeros} accuracy zeros — invented content. Disqualifying. **")

if grades: summarise(grades, key)
else: print("no grades recorded yet")


## 9. What to write up

One page, and these five things:

1. **The table** from section 5, unedited.
2. **Any disqualifying finding** — an invented name, a fabricated date, a
   confident description of a blank page. One clear instance is enough to
   report; a rate is not needed.
3. **A recommendation and a runner-up**, with the condition that would change
   your mind.
4. **What you could not measure.** Whether descriptions are *useful to an
   archivist* is a different question from whether they are correct, and it
   needs an archivist. Say so rather than letting the numbers imply it.
5. **The sample you used** — how many files, of what kinds, from which
   accession. A result without its sample is not reproducible.

Honest limits of this harness, worth stating in the write-up:

- The degraded cases are built by rule, so they test one failure mode
  (empty and truncated input), not all of them.
- Self-consistency at temperature 0 is a weak test; a model can be consistent
  and consistently wrong.
- Nothing here measures whether the intake form actually improves description
  quality, which is a separate experiment worth running: same files, same
  model, with and without the collection context block.
